#### 1. Confirm the Gold tables

In [1]:
gold_tables = [
    "gold_dim_customer",
    "gold_dim_depot",
    "gold_dim_date",
    "gold_fact_rental",
    "gold_fact_billing",
]

for table_name in gold_tables:
    print(
        f"{table_name:<25}",
        spark.table(table_name).count()
    )

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 3, Finished, Available, Finished, False)

gold_dim_customer         600
gold_dim_depot            6
gold_dim_date             30
gold_fact_rental          942
gold_fact_billing         779


##### (a) Average rental duration by asset type

In [2]:
result_a = spark.sql("""
    SELECT
        asset_type,
        ROUND(
            AVG(CAST(rental_duration_days AS DOUBLE)),
            2
        ) AS avg_duration_days,
        COUNT(*) AS rentals
    FROM gold_fact_rental
    WHERE is_returned = 1
    GROUP BY asset_type
    ORDER BY avg_duration_days DESC, asset_type
""")

display(result_a)

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8f9967c-07e2-4636-bab4-4efb2450f8c6)

In [3]:
assert result_a.columns == [
    "asset_type",
    "avg_duration_days",
    "rentals"
]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 5, Finished, Available, Finished, False)

##### (b) Fleet utilisation by depot

In [4]:
result_b = spark.sql("""
    SELECT
        d.depot_name,
        d.fleet_size,
        ROUND(
            SUM(CAST(r.asset_days_in_window AS DOUBLE)),
            1
        ) AS asset_days_used,
        ROUND(
            (
                SUM(CAST(r.asset_days_in_window AS DOUBLE))
                / (d.fleet_size * 30.0)
            ) * 100,
            1
        ) AS utilisation_pct
    FROM gold_fact_rental r
    INNER JOIN gold_dim_depot d
        ON r.depot_code = d.depot_code
    GROUP BY
        d.depot_name,
        d.fleet_size
    ORDER BY utilisation_pct DESC, d.depot_name
""")

display(result_b)

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 243e2fff-d8fd-4f83-9453-a5c056e18c4c)

In [5]:
assert result_b.columns == [
    "depot_name",
    "fleet_size",
    "asset_days_used",
    "utilisation_pct"
]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 7, Finished, Available, Finished, False)

In [6]:
invalid_utilisation = (
    result_b
    .filter(
        (result_b.utilisation_pct < 0)
        | (result_b.utilisation_pct > 100)
    )
    .count()
)

assert invalid_utilisation == 0

print("All depot utilisation values are between 0% and 100%.")

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 8, Finished, Available, Finished, False)

All depot utilisation values are between 0% and 100%.


##### (c) Revenue by payer type

In [7]:
result_c = spark.sql("""
    SELECT
        payer_type,
        COUNT(*) AS bills,
        ROUND(
            SUM(amount_inr),
            0
        ) AS revenue,
        ROUND(
            AVG(amount_inr),
            0
        ) AS avg_bill
    FROM gold_fact_billing
    GROUP BY payer_type
    ORDER BY revenue DESC, payer_type
""")

display(result_c)

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9267d18f-d51a-41c4-87ca-1edf1ce540a0)

In [8]:
assert result_c.columns == [
    "payer_type",
    "bills",
    "revenue",
    "avg_bill"
]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 10, Finished, Available, Finished, False)

In [9]:
payer_type_bill_total = (
    result_c
    .agg({"bills": "sum"})
    .first()[0]
)

print(f"Bills represented: {payer_type_bill_total}")

assert payer_type_bill_total == 779

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 11, Finished, Available, Finished, False)

Bills represented: 779


##### (d) Revenue by depot

In [10]:
result_d = spark.sql("""
    SELECT
        d.depot_name,
        COUNT(*) AS bills,
        ROUND(
            SUM(b.amount_inr),
            0
        ) AS revenue,
        ROUND(
            SUM(b.amount_inr)
            / COUNT(DISTINCT r.rental_id),
            0
        ) AS revenue_per_rental
    FROM gold_fact_billing b
    INNER JOIN gold_fact_rental r
        ON b.rental_id = r.rental_id
    INNER JOIN gold_dim_depot d
        ON r.depot_code = d.depot_code
    GROUP BY d.depot_name
    ORDER BY revenue DESC, d.depot_name
""")

display(result_d)

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad03573b-e2d5-4f66-b61a-12463a20debc)

In [11]:
assert result_d.columns == [
    "depot_name",
    "bills",
    "revenue",
    "revenue_per_rental"
]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 13, Finished, Available, Finished, False)

###### Reconcile total revenue

In [12]:
billing_fact_total = spark.sql("""
    SELECT
        SUM(amount_inr) AS total_revenue
    FROM gold_fact_billing
""").first()["total_revenue"]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 14, Finished, Available, Finished, False)

In [13]:
depot_revenue_total = spark.sql("""
    SELECT
        SUM(b.amount_inr) AS total_revenue
    FROM gold_fact_billing b
    INNER JOIN gold_fact_rental r
        ON b.rental_id = r.rental_id
    INNER JOIN gold_dim_depot d
        ON r.depot_code = d.depot_code
""").first()["total_revenue"]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 15, Finished, Available, Finished, False)

In [14]:
print(f"Billing fact total:    ₹{billing_fact_total:,.2f}")
print(f"Depot revenue total:   ₹{depot_revenue_total:,.2f}")
print(
    "Revenue in crore:     "
    f"₹{float(depot_revenue_total) / 10_000_000:.2f} crore"
)

assert billing_fact_total == depot_revenue_total

print("Revenue reconciliation passed.")

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 16, Finished, Available, Finished, False)

Billing fact total:    ₹49,775,400.42
Depot revenue total:   ₹49,775,400.42
Revenue in crore:     ₹4.98 crore
Revenue reconciliation passed.


##### (e) Share of rentals by rental type

In [17]:
result_e = spark.sql("""
    WITH rental_type_counts AS (
        SELECT
            rental_type,
            COUNT(*) AS rentals
        FROM gold_fact_rental
        GROUP BY rental_type
    )
    SELECT
        rental_type,
        rentals,
        ROUND(
            rentals * 100.0
            / SUM(rentals) OVER (),
            1
        ) AS pct
    FROM rental_type_counts
    ORDER BY rentals DESC, rental_type
""")

display(result_e)

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc6636fa-6576-43b6-baa0-04e462026d09)

In [18]:
assert result_e.columns == [
    "rental_type",
    "rentals",
    "pct"
]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 20, Finished, Available, Finished, False)

In [19]:
rental_mix_summary = (
    result_e
    .agg(
        F.sum("rentals").alias("total_rentals"),
        F.sum("pct").alias("total_pct")
    )
    .first()
)

print(
    "Total rentals:",
    rental_mix_summary["total_rentals"]
)

print(
    "Rounded percentage total:",
    rental_mix_summary["total_pct"]
)

assert rental_mix_summary["total_rentals"] == 942
assert result_e.count() == 2

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 21, Finished, Available, Finished, False)

Total rentals: 942
Rounded percentage total: 100.0


##### (f) Machines currently out by depot

In [20]:
result_f = spark.sql("""
    SELECT
        d.depot_name,
        COUNT(*) AS currently_out
    FROM gold_fact_rental r
    INNER JOIN gold_dim_depot d
        ON r.depot_code = d.depot_code
    WHERE r.is_returned = 0
    GROUP BY d.depot_name
    ORDER BY currently_out DESC, d.depot_name
""")

display(result_f)

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7f6c4310-b3ae-4caa-9982-5990deb4845f)

In [21]:
assert result_f.columns == [
    "depot_name",
    "currently_out"
]

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 23, Finished, Available, Finished, False)

In [22]:
gold_currently_out = (
    result_f
    .agg(
        F.sum("currently_out").alias("total")
    )
    .first()["total"]
)

silver_currently_out = (
    spark.table("silver_rentals")
    .filter(F.col("checkin_ts").isNull())
    .count()
)

print(
    f"Gold currently-out total:   {gold_currently_out}"
)

print(
    f"Silver still-out total:     {silver_currently_out}"
)

assert gold_currently_out == silver_currently_out
assert gold_currently_out == 175

print("Currently-out reconciliation passed.")

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 24, Finished, Available, Finished, False)

Gold currently-out total:   175
Silver still-out total:     175
Currently-out reconciliation passed.


##### Complete acceptance validation

In [23]:
expected_columns = {
    "a": [
        "asset_type",
        "avg_duration_days",
        "rentals"
    ],
    "b": [
        "depot_name",
        "fleet_size",
        "asset_days_used",
        "utilisation_pct"
    ],
    "c": [
        "payer_type",
        "bills",
        "revenue",
        "avg_bill"
    ],
    "d": [
        "depot_name",
        "bills",
        "revenue",
        "revenue_per_rental"
    ],
    "e": [
        "rental_type",
        "rentals",
        "pct"
    ],
    "f": [
        "depot_name",
        "currently_out"
    ]
}

actual_results = {
    "a": result_a,
    "b": result_b,
    "c": result_c,
    "d": result_d,
    "e": result_e,
    "f": result_f
}

for result_name, dataframe in actual_results.items():
    assert dataframe.columns == expected_columns[result_name]

assert payer_type_bill_total == 779
assert rental_mix_summary["total_rentals"] == 942
assert billing_fact_total == depot_revenue_total
assert gold_currently_out == 175
assert silver_currently_out == 175

print("All six business-query acceptance criteria passed.")

StatementMeta(, 36498cb2-7a78-4deb-a850-e019261c8e84, 25, Finished, Available, Finished, False)

All six business-query acceptance criteria passed.


**Intepretation of (d) and (f) in plain business language: which depot earns the most, which earns the most per rental, and how many machines are on site right now. Name the null-safety point, that the currently-out figure would have been zero if the Silver filter had been careless.**

Hadapsar Depot generated the highest total revenue at ₹10,785,755, making it BuildMate’s largest revenue contributor during the reporting period. However, Talegaon Depot earned the most revenue per rental, at approximately ₹90,920 per rental, indicating a higher-value rental mix despite having fewer bills. Across all depots, 175 machines are currently out on customer sites, with Chakan having the most at 45, followed by Hadapsar with 40. This currently-out figure is reliable because the Silver transformation used a null-safe filter that preserved rentals with blank check-in timestamps. Had the careless filter(~bad) logic been used, those active rentals would have been silently removed and the reported number of machines currently on site would have incorrectly appeared as zero.

# BuildMate Rentals — Gold Business Findings

## Purpose

The Gold star schema combines the rental-management, billing, customer,
and depot datasets into one conformed analytical model. This allows
BuildMate to answer operational and financial questions that no individual
source system can answer by itself.

## 1. Rental duration by machine type

Returned rentals were grouped by asset type and their average completed
rental duration was calculated.

The machine type with the longest average rental duration was
**[LDR]**, at **[5.24] days**, based on **[100] returned rentals**.


Currently-out rentals were excluded from this calculation because their
final durations are not yet known. Their duration remains NULL rather than
being represented incorrectly as zero.

## 2. Fleet utilisation by depot

Fleet utilisation was calculated as:

asset days used / (fleet size × 30 reporting days)

The most heavily utilised depot was **[Talegaon Depot]**, with utilisation of
**[83.8]%**. This indicates that it consumed the largest share of its
available fleet-days during the June reporting window.

The asset-days measure was clipped to the reporting window, so no rental
could contribute more than thirty days.

## 3. Revenue by payer type

Revenue was grouped by the standardised payer types: direct, contract,
corporate, and prepaid.

The highest-revenue payer type was **[direct]**, contributing
approximately **₹[16394613]** across **[220] bills**, with an average bill
of **₹[74521]**.

## 4. Revenue by depot

Billing was connected to rentals through rental_id, and rentals were then
connected to the depot dimension.

This produced the depot revenue figure that neither the billing system nor
the rental-management system could provide independently.
Total matched revenue was **₹[49,775,400.42]**, approximately **[4.98] crore**.
This reconciled exactly to the total in gold_fact_billing.

The highest-revenue depot was **[Hadapsar Depot]**, generating approximately
**₹[10785755]**.

## 5. Rental priority mix

The rental mix consisted of:

- Standard: **[681] rentals ([72.3]%)**
- Priority: **[261] rentals ([27.7]%)**

This shows the proportion of the business requiring priority handling
compared with normal rental activity.

## 6. Machines currently out

There were **175 machines currently out on site** at the reporting point.

This total reconciled exactly to the 175 blank check-in records preserved
by the null-safe Silver quality filter. A careless filter would have
silently removed all of these active rentals.

The depot with the most machines currently out was **[Chakan Depot]**, with
**[45] machines**.

## Conclusion

The conformed Gold model provides one trusted view of BuildMate's rental
operations and financial performance. It connects depot capacity, machine
activity, rental duration, customer behaviour, and billing revenue while
preserving active rentals and reconciling all totals to the cleaned Silver
layer.
